# Verify: Graph Edges Match

This notebook checks if the edges displayed in visualization match the actual graph

In [1]:
import sys
sys.path.insert(0, '/Users/tduani/PycharmProjects/distributed_node_matching')

import networkx as nx
import random
from src.graph import GraphManager

# Same parameters as visualization.ipynb
NODE_COUNT = 10
EDGE_DENSITY = 0.4
GRAPH_TYPE = 'random'
SEED = 42

random.seed(SEED)

# Generate graph EXACTLY as visualization.ipynb does
graph = GraphManager.create_empty_graph()
for i in range(1, NODE_COUNT + 1):
    graph.add_vertex(i)

vertex_list = list(range(1, NODE_COUNT + 1))
edges_added = set()

max_possible_edges = NODE_COUNT * (NODE_COUNT - 1) // 2
target_edges = int(max_possible_edges * EDGE_DENSITY)

print(f"Generating {target_edges} edges from {max_possible_edges} possible...\n")

edge_creation_order = []
while len(edges_added) < target_edges:
    u = random.choice(vertex_list)
    v = random.choice(vertex_list)
    if u != v and (min(u, v), max(u, v)) not in edges_added:
        weight = random.uniform(0.0, 1.0)
        graph.add_edge(u, v, weight)
        edge_key = (min(u, v), max(u, v))
        edges_added.add(edge_key)
        edge_creation_order.append(edge_key)

print(f"✅ Graph created with {len(edges_added)} edges")

Generating 18 edges from 45 possible...

✅ Graph created with 18 edges


In [2]:
# Now convert to NetworkX exactly as visualization.ipynb does in Cell 6
G = nx.Graph()
for v in graph.vertices():
    G.add_node(v)
for u, v in graph._graph.edges():
    weight = graph.get_edge_weight(u, v)
    G.add_edge(u, v, weight=weight)

print(f"NetworkX graph has {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")
print(f"\nEdges in NetworkX graph:")
nx_edges = sorted(list(G.edges()))
for u, v in nx_edges:
    print(f"  {u} -- {v}")

NetworkX graph has 10 nodes and 18 edges

Edges in NetworkX graph:
  1 -- 2
  1 -- 3
  1 -- 4
  1 -- 8
  1 -- 10
  2 -- 3
  2 -- 4
  2 -- 6
  2 -- 7
  2 -- 9
  3 -- 6
  4 -- 5
  4 -- 10
  5 -- 6
  5 -- 8
  5 -- 10
  6 -- 10
  7 -- 9


In [3]:
# Compare
print("\n" + "="*70)
print("COMPARISON")
print("="*70)

source_edges = set(edges_added)
nx_edges_set = set(G.edges())

print(f"\nGraphManager edges: {len(source_edges)}")
print(f"NetworkX edges: {len(nx_edges_set)}")

if source_edges == nx_edges_set:
    print("\n✅ EDGES MATCH! Both have the same edges.")
else:
    missing = source_edges - nx_edges_set
    extra = nx_edges_set - source_edges
    
    if missing:
        print(f"\n❌ Missing from NetworkX ({len(missing)}):")
        for edge in sorted(missing):
            print(f"  {edge}")
    
    if extra:
        print(f"\n❌ Extra in NetworkX ({len(extra)}):")
        for edge in sorted(extra):
            print(f"  {edge}")


COMPARISON

GraphManager edges: 18
NetworkX edges: 18

✅ EDGES MATCH! Both have the same edges.


In [4]:
# Print the actual set of edges you see
print("\n" + "="*70)
print("EDGES IN YOUR GRAPH")
print("="*70)
print(f"\nYour edges set (from your message):")
your_edges = {(4, 9), (5, 7), (5, 10), (1, 6), (1, 3), (4, 5), (5, 6), (3, 6), (2, 4), (1, 2), (2, 7), (2, 10), (1, 8), (7, 9), (6, 10), (4, 10), (5, 8), (1, 4), (2, 3), (2, 9), (2, 6), (1, 10)}

print(f"\nComparing YOUR edges with WHAT WAS GENERATED:")
print(f"Your edges: {len(your_edges)}")
print(f"Generated edges: {len(source_edges)}")

if your_edges == source_edges:
    print("\n✅ YOUR EDGES MATCH WHAT WAS GENERATED!")
else:
    missing = source_edges - your_edges
    extra = your_edges - source_edges
    
    if missing:
        print(f"\n❌ In generated but NOT in your graph ({len(missing)}):")
        for edge in sorted(missing):
            print(f"  {edge}")
    
    if extra:
        print(f"\n❌ In your graph but NOT generated ({len(extra)}):")
        for edge in sorted(extra):
            print(f"  {edge}")


EDGES IN YOUR GRAPH

Your edges set (from your message):

Comparing YOUR edges with WHAT WAS GENERATED:
Your edges: 22
Generated edges: 18

❌ In your graph but NOT generated (4):
  (1, 6)
  (2, 10)
  (4, 9)
  (5, 7)
